### Experimentation Purpose Only

## Train on CelebA

In [1]:
from train import Trainer
from config import config
from model import UNet
from modules import NoiseInjectionPipeline
from denoising_diffusion import NoisePredictionPipeline, DDPM

# Forward diffusion
noise_injection_pipeline = NoiseInjectionPipeline(
    config=config,
    random=True
)

# Noise prediction backbone
unet = UNet(
    in_channels = config["in_channels"], # because not it will operate in latent space and not in RGB Space
    out_channels = config["out_channels"],
    channel_multipliers = (1, 2, 4, 8),
    time_embd_dim = config["time_embed"],
    base_channels = 64
)

# Noise prediction pipeline
noise_prediction_pipeline = NoisePredictionPipeline(
    config=config,
    noise_prediction_model=unet
)

# Complete DDPM
ddpm = DDPM(
    noise_injection_pipieline=noise_injection_pipeline,
    noise_prediction_pipeline=noise_prediction_pipeline
)

# trainer = Trainer(
#     ddpm=ddpm,
#     config=config,

#     dataset_path="../data/celeba/celeba_latents.pt",

#     batch_size = config["batch_size"],
#     image_size = config["image_size"],

#     epochs = 350,
#     warmup_epochs = 10,
#     learning_rate = 1e-4,
#     use_min_snr_loss = config["use_min_snr_loss"],

#     validation_split = 0.05,

#     checkpoint_dir="./checkpoint/celeba_256",
#     checkpoint_every = 10,

#     project_name="DiffusionEngine",
#     run_name="DDPM-CelebA-HQ-256x256",

#     num_workers = 4,
#     use_amp = False
# )
# trainer.resume(checkpoint_path = "./checkpoint/celeba_256/latest.pt")
# trainer.fit()

##

In [2]:
# import torch
# from sample import sample
# import matplotlib.pyplot as plt
# from diffusers import AutoencoderKL

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# vae = AutoencoderKL.from_pretrained(
#         config["vae_model"]
#     ).to(device)

# vae.eval()
# vae.requires_grad_(False)

# generated_image = sample(ddpm, config, device, batch_size=config["batch_size"], vae = vae)
# generated_image = (generated_image + 1.0) / 2.0 # Convert [-1, 1] -> [0, 1]
# generated_image = generated_image.clamp(0, 1) # for visualization in matplotlib

# # Visualize

# # Convert BCHW -> HWC
# image = generated_image[0].permute(1, 2, 0).cpu().numpy()

# plt.figure(figsize=(6, 6))
# plt.imshow(image)
# plt.axis("off")
# plt.title("Generated Image")
# plt.show()